# Word2Vec
"Efficient Estimation of Word Representations in Vector Space" and "Distributed Representations of Words and Phrases and their Compositionality" (Mikolov et al., 2013)

**Idea:** a word is characterized by the company it keeps. Train a tiny model to predict which words appear **near** a given word; the model has no hidden layer, so all it can learn is one vector per word. Words used in similar contexts end up with similar vectors.

### Word2Vec vs other word representations
| | One-hot | Word2Vec (skip-gram) | Word2Vec (CBOW) | GloVe | BERT (`bert.ipynb`) |
|---|---|---|---|---|---|
| Vector | $V$-dim, sparse | $D$-dim, dense | $D$-dim, dense | $D$-dim, dense | $D$-dim, dense |
| Trained by | - | center word -> predict each context word | averaged context -> predict center word | factorize the co-occurrence matrix | masked LM with a deep encoder |
| Similar words close? | no (all orthogonal) | yes | yes | yes | yes |
| Depends on the sentence? | no | **no** (one vector per word) | no | no | **yes** (contextual) |
| Model depth | - | 0 hidden layers | 0 hidden layers | 0 | 12+ Transformer layers |
| Cost | - | very cheap (billions of words/day on CPU) | cheapest | cheap | expensive |

Here: skip-gram with negative sampling (SGNS) from scratch, including the paper's three tricks (subsampling of frequent words, the $\text{unigram}^{3/4}$ noise distribution, dynamic window), trained on the Enron emails (ham and spam) used in `bert.ipynb`.

Shape flow: `center (B,), context (B,), negatives (B, K) -> V[center] (B, D), U[context] (B, D), U[negatives] (B, K, D) -> dot products (B,), (B, K) -> log-sigmoid loss (scalar)`

In [ ]:
import re
import math
from collections import Counter

import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42)

## 1. Data - Enron emails as one long word stream
33,716 emails (subject + body), lowercased, letters only. Words seen fewer than 10 times are dropped (the paper uses a minimum count of 5). The emails are concatenated into one stream; a window occasionally straddles two emails, which adds a little noise and keeps the code simple.

No labels are used: like MLM in `bert.ipynb`, this is **self-supervised** - the text itself provides the targets.

In [ ]:
# the original Enron-Spam csv (github.com/MWiechmann/enron_spam_data); in the older local enron_spam_data.csv the spam rows are 6 ham emails repeated
df = pd.read_csv("../datasets/enron_spam_data_original.csv")
texts = (df["Subject"].fillna("") + " " + df["Message"].fillna("")).str.lower().str.slice(0, 5000)

words = [w for t in texts for w in re.findall(r"[a-z]+", t)]
counts = Counter(words)
min_count = 10
itos = [w for w, c in counts.most_common() if c >= min_count]     # sorted by frequency
stoi = {w: i for i, w in enumerate(itos)}
vocab_size = len(itos)

ids = torch.tensor([stoi[w] for w in words if w in stoi])          # (N,) the corpus as word ids
freq = torch.tensor([counts[w] for w in itos], dtype=torch.float)  # (V,) raw counts
del words, texts

print(f"corpus: {len(ids) / 1e6:.2f}M words | vocab: {vocab_size}")
print("most frequent:", itos[:15])
print("top 10 words make up", f"{(freq[:10].sum() / freq.sum()).item():.1%}", "of the corpus")

## 2. Subsampling of frequent words
`the`, `to`, `and` appear next to everything, so the pair (`gas`, `the`) says almost nothing about `gas`, yet such pairs dominate the data. Each occurrence of a word is **dropped** with a probability that grows with its frequency. This is done once, before the windows are formed, so the remaining words also get effectively wider windows.

### Formula: Subsampling
$$P_{keep}(w) = \min\!\left(1,\ \sqrt{\frac{t}{f(w)}}\right)$$
- $f(w)$: fraction of the corpus that is word $w$
- $t = 10^{-4}$: threshold (the paper suggests $10^{-5}$ for billions of words; larger for small corpora)
- words rarer than $t$ are always kept; a word with $f = 0.04$ (`the`) is kept 5% of the time
- this is the paper's formula $P_{drop} = 1 - \sqrt{t / f}$; the C code uses a slightly different one

In [ ]:
t = 1e-4
f = freq / freq.sum()                                   # (V,)
p_keep = (t / f).sqrt().clamp(max=1.0)                  # (V,)

keep = torch.rand(len(ids)) < p_keep[ids]               # (N,) one coin flip per occurrence
corpus = ids[keep].to(device)                           # (N',)
print(f"kept {len(corpus) / 1e6:.2f}M of {len(ids) / 1e6:.2f}M words")
for w in ["the", "enron", "meeting", "pipeline", "viagra"]:
    print(f"   {w:10s} count {int(freq[stoi[w]]):7d} | P_keep {p_keep[stoi[w]].item():.3f}")

## 3. Training pairs: (center, context)
For every position, the words within a window around it are its positive contexts. The window size is **dynamic**: for each center a size $R \in \{1..W\}$ is drawn, so nearby words are sampled more often than distant ones.

Instead of materializing every pair (tens of millions), each step samples `batch_size` random centers and one context for each.

### Formula: Dynamic window
$$R \sim \text{Uniform}\{1, \dots, W\}, \qquad \text{contexts}(i) = \{w_{i+j} : -R \le j \le R,\ j \ne 0\}$$
- $W = 5$: max window
- a word at distance $d$ is inside the window with probability $(W - d + 1) / W$: distance 1 always, distance 5 one time in five
- we draw $R$, then one offset $j$ uniformly from $\{1..R\}$ with a random sign

In [ ]:
window = 5       # W
batch_size = 4096

def get_batch():
    # -> center (B,), context (B,) word ids
    i = torch.randint(window, len(corpus) - window, (batch_size,), device=device)   # (B,) positions
    R = torch.randint(1, window + 1, (batch_size,), device=device)                   # (B,) window size per center
    j = (torch.rand(batch_size, device=device) * R).long() + 1                       # (B,) offset in 1..R
    sign = torch.randint(0, 2, (batch_size,), device=device) * 2 - 1                 # (B,) left or right
    return corpus[i], corpus[i + sign * j]

c, o = get_batch()
print([(itos[a], itos[b]) for a, b in zip(c[:8].tolist(), o[:8].tolist())])

## 4. Skip-gram with negative sampling
The original skip-gram objective is a softmax over the whole vocabulary for every pair, which costs $O(V)$ per example.

### Formula: Skip-gram softmax (too expensive)
$$p(o \mid c) = \frac{\exp(u_o^T v_c)}{\sum_{w=1}^{V} \exp(u_w^T v_c)}$$
- $v_c \in \mathbb{R}^D$: **input** ("center") vector of word $c$, a row of $V_{in} \in \mathbb{R}^{V \times D}$
- $u_o \in \mathbb{R}^D$: **output** ("context") vector of word $o$, a row of $U_{out} \in \mathbb{R}^{V \times D}$
- every word has two vectors; the denominator touches all $V$ output vectors per training pair

Negative sampling replaces it with $K + 1$ binary classifications: "is this (center, context) pair real, or is the context a random word?"

### Formula: Negative-sampling loss
$$\mathcal{L} = -\log \sigma(u_o^T v_c) - \sum_{k=1}^{K} \log \sigma(-u_{w_k}^T v_c), \qquad w_k \sim P_n(w)$$
- $\sigma$: sigmoid; first term pulls the true context's vector toward $v_c$
- $w_k$: $K = 5$ **negative** words drawn from the noise distribution; the second term pushes them away
- cost per pair: $K + 1$ dot products instead of $V$
- a uniform-random model scores $(K + 1) \log 2 \approx 4.16$

### Formula: Noise distribution
$$P_n(w) = \frac{\text{count}(w)^{3/4}}{\sum_{w'} \text{count}(w')^{3/4}}$$
- the $3/4$ power sits between the unigram distribution (power 1, negatives are almost all `the`) and uniform (power 0, negatives are mostly rare words that are too easy)

In [ ]:
class SkipGram(nn.Module):
    # center (B,), context (B,), negatives (B, K) -> loss (scalar)
    def __init__(self, vocab_size, dim):
        super().__init__()
        self.in_embed = nn.Embedding(vocab_size, dim)     # V_in:  the vectors we keep
        self.out_embed = nn.Embedding(vocab_size, dim)    # U_out: context vectors, usually discarded
        nn.init.uniform_(self.in_embed.weight, -0.5 / dim, 0.5 / dim)   # init as in the C code
        nn.init.zeros_(self.out_embed.weight)

    def forward(self, center, context, negatives):
        v = self.in_embed(center)                                       # (B, D)
        u_pos = self.out_embed(context)                                 # (B, D)
        u_neg = self.out_embed(negatives)                               # (B, K, D)
        pos = (u_pos * v).sum(-1)                                       # (B,)   u_o . v_c
        neg = (u_neg @ v[:, :, None]).squeeze(-1)                       # (B, K, D) @ (B, D, 1) -> (B, K)
        return -(F.logsigmoid(pos) + F.logsigmoid(-neg).sum(-1)).mean() # scalar

dim = 100          # D
num_neg = 5        # K
noise = (freq ** 0.75).to(device)                                       # (V,) unnormalized P_n; multinomial normalizes

model = SkipGram(vocab_size, dim).to(device)
neg = torch.multinomial(noise, batch_size * num_neg, replacement=True).view(batch_size, num_neg)
print(f"params: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M | initial loss {model(c, o, neg).item():.4f} (expected {(num_neg + 1) * math.log(2):.4f})")
print("negatives:", [itos[i] for i in neg[0].tolist()])

## 5. Training
The paper trains with plain SGD and a linearly decaying learning rate, one pair at a time on CPU threads. On a GPU with large batches, **Adam** with the same linear decay converges in far fewer steps.

Negatives are drawn independently of the center, so a negative is occasionally the true context; with $V$ in the tens of thousands this is rare and ignored (as in the original).

In [ ]:
max_iters = 30000
lr = 2e-3
log_every = 3000

optimizer = torch.optim.Adam(model.parameters(), lr=lr)
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lambda s: 1 - s / max_iters)   # linear decay to 0

losses, run = [], 0.0
for it in range(1, max_iters + 1):
    center, context = get_batch()                                                         # (B,), (B,)
    negatives = torch.multinomial(noise, batch_size * num_neg, replacement=True).view(batch_size, num_neg)  # (B, K)
    loss = model(center, context, negatives)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    scheduler.step()

    run += loss.item()
    if it % log_every == 0:
        losses.append((it, run / log_every))
        print(f"iter {it:6d} | loss {run / log_every:.4f}")
        run = 0.0

plt.plot(*zip(*losses), marker="o")
plt.xlabel("iter"); plt.ylabel("SGNS loss"); plt.show()

## 6. Nearest neighbors
The word vectors are the rows of the **input** embedding. Similarity is the cosine of the angle between two vectors (vector length mostly reflects word frequency, so it is normalized away).

### Formula: Cosine similarity
$$\text{sim}(a, b) = \frac{v_a^T v_b}{\lVert v_a \rVert\, \lVert v_b \rVert}$$
- 1 = same direction, 0 = unrelated, -1 = opposite

In [ ]:
E = F.normalize(model.in_embed.weight.detach(), dim=-1)      # (V, D) unit-length word vectors

def neighbors(word, k=8):
    sims = E @ E[stoi[word]]                                  # (V,) cosine similarity to every word
    top = sims.topk(k + 1).indices[1:]                        # drop the word itself
    return [itos[i] for i in top.tolist()]

for w in ["monday", "january", "gas", "meeting", "money", "viagra", "houston", "she", "three", "buy"]:
    print(f"{w:10s} -> {neighbors(w)}")

## 7. Analogies: vector arithmetic
The paper's best-known result: relations between words show up as (roughly) constant **offsets** between their vectors, so `king - man + woman` lands near `queen`.

### Formula: Analogy $a : b :: c : ?$
$$d^* = \arg\max_{d \notin \{a, b, c\}} \text{sim}\big(v_d,\ v_b - v_a + v_c\big)$$
- $v_b - v_a$: the relation (e.g. singular -> plural, he -> she)
- the three question words are excluded from the answer

With ~6M words of business email and spam, only relations that are frequent in this corpus (grammar, days, months, numbers) can be expected to work; the paper trained on 1-6 billion words of news.

In [ ]:
def analogy(a, b, c, k=4):
    q = F.normalize(E[stoi[b]] - E[stoi[a]] + E[stoi[c]], dim=0)   # (D,)
    sims = E @ q                                                    # (V,)
    sims[[stoi[a], stoi[b], stoi[c]]] = -1                          # exclude the question words
    return [itos[i] for i in sims.topk(k).indices.tolist()]

for a, b, c in [("he", "his", "she"), ("he", "him", "they"), ("is", "was", "are"), ("day", "days", "week"),
                ("buy", "bought", "sell"), ("monday", "tuesday", "wednesday"), ("january", "february", "march"), ("one", "two", "three")]:
    print(f"{a} : {b} :: {c} : {analogy(a, b, c)}")

## 8. A 2D picture
Project the 100-dim vectors of a few word groups onto their first two principal components (PCA, see `PCA_SVD_Eigendecomposition.ipynb`). Words of the same kind should cluster.

In [ ]:
groups = {
    "days": ["monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday"],
    "months": ["january", "february", "march", "april", "june", "july", "august", "october", "december"],
    "numbers": ["one", "two", "three", "four", "five", "six", "ten"],
    "energy": ["gas", "oil", "power", "pipeline", "electricity", "energy", "fuel"],
    "spam": ["viagra", "cialis", "pills", "prescription", "meds", "pharmacy", "drugs"],
    "pronouns": ["he", "she", "they", "we", "you", "i"],
}
sel = [w for ws in groups.values() for w in ws if w in stoi]
X = E[[stoi[w] for w in sel]]                         # (n, D)
X = X - X.mean(0)
_, _, Vt = torch.linalg.svd(X, full_matrices=False)   # rows of Vt = principal directions
xy = (X @ Vt[:2].T).cpu()                             # (n, 2)

plt.figure(figsize=(9, 7))
i = 0
for name, ws in groups.items():
    ws = [w for w in ws if w in stoi]
    pts = xy[i:i + len(ws)]
    plt.scatter(pts[:, 0], pts[:, 1], label=name)
    for (px, py), w in zip(pts.tolist(), ws):
        plt.annotate(w, (px, py), fontsize=8, xytext=(3, 3), textcoords="offset points")
    i += len(ws)
plt.legend(); plt.show()

## Notes
- The whole model is two embedding tables and a dot product. Levy & Goldberg (2014) showed SGNS implicitly factorizes a shifted **PMI** matrix: $u_o^T v_c \approx \text{PMI}(c, o) - \log K$. Word2Vec is matrix factorization of co-occurrence statistics done by SGD, which is why GloVe (explicit factorization) gives similar vectors.
- The three tricks matter more than the architecture: subsampling and the $3/4$ noise distribution account for most of the quality gap to earlier count-based methods.
- **CBOW** is the mirror image: average the context vectors and predict the center word. Faster, slightly worse on rare words. **Hierarchical softmax** is the paper's other alternative to the full softmax (a binary tree over the vocab, $O(\log V)$ per pair).
- One vector per word cannot separate senses (`bank`, or here `march` the month vs the verb). Contextual models (ELMo, then BERT) fixed this by computing the vector from the whole sentence; their *input* embedding table is still exactly this kind of lookup.
- **fastText** (Bojanowski et al., 2017) adds character n-gram vectors, so unseen and misspelled words still get an embedding.
- The same objective is reused well outside text: item2vec / product embeddings in recommenders (`collaborative_filtering.ipynb`), node2vec for graphs, and, with in-batch negatives and a softmax, the **contrastive** loss behind CLIP and sentence-embedding models.